<a href="https://colab.research.google.com/github/susara20010420/Workplace-Safety-Insights-from-the-Industrial-Safety-Health-Analytics-Dataset/blob/main/v5_Bayesian_Methodology_Companion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This cell provides a title and brief description for the notebook, indicating its focus on a Bayesian Severity Companion Model and its methodology-aligned template.

# Bayesian Severity Companion Model
Methodology-aligned template.

This cell installs the necessary Python packages for the project, including `pymc` and `arviz` for Bayesian modeling, `sentence-transformers` for text embeddings, and `scikit-learn` for machine learning utilities.

In [ ]:
# Install
!pip -q install pymc arviz sentence-transformers scikit-learn

This cell imports various libraries essential for data manipulation, Bayesian modeling, text processing, and file handling. It also mounts Google Drive to access project files and loads the training, testing, and transformer predictions datasets, along with class weights, from the specified project folder.

In [ ]:
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
import json
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA

from google.colab import drive
drive.mount('/content/drive')
import os

project_folder="/content/drive/MyDrive/Work Place Safety Insights"

train_df=pd.read_csv(f"{project_folder}/train.csv")
test_df=pd.read_csv(f"{project_folder}/test.csv")
transformer_predictions=pd.read_csv(f"{project_folder}/transformer_predictions.csv")

with open(f"{project_folder}/class_weights.json") as f:
    class_weights=json.load(f)


Mounted at /content/drive


This cell serves as a section header, indicating that the following cells will deal with MiniLM embeddings and Principal Component Analysis (PCA).

## MiniLM Embeddings + PCA

This cell initializes a `SentenceTransformer` model to generate embeddings for the 'description' column in both the training and testing datasets. It then applies PCA to reduce the dimensionality of these embeddings to 20 components, preparing the data for the Bayesian model. Finally, it extracts the target labels for training and testing.

In [ ]:
encoder=SentenceTransformer("all-MiniLM-L6-v2")

train_emb=encoder.encode(train_df["description"].tolist(),show_progress_bar=True)
test_emb=encoder.encode(test_df["description"].tolist(),show_progress_bar=True)

# Reduced PCA components from 100 to 20 as suggested to address overfitting with small dataset
pca=PCA(n_components=20,random_state=42)
X_train=pca.fit_transform(train_emb)
X_test=pca.transform(test_emb)

y_train=train_df["severity_label"].values
y_test=test_df["severity_label"].values

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

This cell is a section header, indicating that the subsequent cells will set up hierarchical indices for the dataset.

## Hierarchical indices

This cell creates numerical hierarchical indices for 'Industry Sector' and 'Local' (plant) columns in both the training and testing dataframes using `pd.factorize`. These indices will be used in the hierarchical Bayesian model to group observations by sector and plant.

In [ ]:
train_df["sector_idx"],sector_names=pd.factorize(train_df["Industry Sector"])
test_df["sector_idx"]=pd.Categorical(test_df["Industry Sector"],categories=sector_names).codes

train_df["plant_idx"],plant_names=pd.factorize(train_df["Local"])
test_df["plant_idx"]=pd.Categorical(test_df["Local"],categories=plant_names).codes


This cell is a section header, introducing the Hierarchical Ordinal Bayesian Model that will be defined and fitted in the following code block.

## Hierarchical Ordinal Bayesian Model

This cell defines and fits a hierarchical ordinal Bayesian model using PyMC. It specifies priors for the model parameters (beta coefficients, plant effects, and cutpoints) and uses an ordered logistic likelihood function. The model is then sampled using MCMC to obtain posterior distributions, and posterior predictive samples are generated.

In [ ]:
from pymc.distributions.transforms import ordered

n_features=X_train.shape[1] # This will now be 20 due to PCA change
n_classes=len(np.unique(y_train))
n_sectors=len(sector_names) # Kept for potential future use or consistency
n_plants=len(plant_names)

with pm.Model() as ordinal_model:
    # Non-centered beta with better priors (HalfNormal(0.5) instead of HalfCauchy(1))
    tau = pm.HalfNormal("tau", 0.5)
    lam = pm.HalfNormal("lam", 0.5, shape=n_features)
    beta_raw = pm.Normal("beta_raw", 0, 1, shape=n_features)
    beta = pm.Deterministic("beta", beta_raw * tau * lam)

    # Hierarchical effects (simplified to plant_effect only with tighter prior)
    plant_sd = pm.HalfNormal("plant_sd", 0.5)
    plant_offset = pm.Normal("plant_offset", 0, 1, shape=n_plants)
    plant_effect = pm.Deterministic("plant_effect", plant_offset * plant_sd)

    # Linear predictor without sector_effect for simplification
    eta = pm.math.dot(X_train, beta) + plant_effect[train_df["plant_idx"].values]

    # Reverting to explicit ordered cutpoints using cumulative sums for robust initialization
    # The 'ordered' transform was causing initialization issues.
    cutpoints_0 = pm.Normal("cutpoints_0", mu=0, sigma=2)
    # Using HalfNormal(0.5) for a tighter prior on differences
    cutpoint_diffs = pm.HalfNormal("cutpoint_diffs", sigma=0.5, shape=n_classes - 2)
    cutpoints = pm.Deterministic("cutpoints",
        pm.math.concatenate([[cutpoints_0], cutpoints_0 + pm.math.cumsum(cutpoint_diffs)]))

    severity = pm.OrderedLogistic(
        "severity",
        eta=eta,
        cutpoints=cutpoints,
        observed=y_train)

    # Increased target_accept, tune, max_treedepth and set cores=1 for better convergence
    trace = pm.sample(1000, tune=2000, target_accept=0.99, max_treedepth=15, return_inferencedata=True, cores=1)

    posterior_pred = pm.sample_posterior_predictive(trace)

Output()

Output()

This cell marks a section for summarizing the posterior distributions obtained from the Bayesian model.

## Posterior summaries

This cell calculates and displays a summary of the posterior distributions of the model parameters using `arviz.summary`. It also saves this summary to a CSV file for later analysis.

In [ ]:
summary=az.summary(trace,hdi_prob=0.95)
summary.to_csv(f"{project_folder}/posterior_summary.csv")
display(summary.head())


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
beta[0],-3.924,0.296,-4.530,-3.368,0.006,0.006,2233.0,1789.0,1.0
beta[1],1.618,0.471,0.705,2.527,0.010,0.009,2248.0,1517.0,1.0
beta[2],4.800,0.498,3.864,5.789,0.011,0.010,2248.0,1659.0,1.0
beta[3],2.045,0.716,0.772,3.581,0.016,0.015,2096.0,1434.0,1.0
beta[4],-0.298,0.350,-1.027,0.308,0.008,0.006,1543.0,1976.0,1.0


This cell introduces the section for performing posterior predictions on the test set, highlighting that this is an example and can be extended for full production inference using `pm.Data`.

## Posterior prediction on test set (replace with full predictive routine as needed)

This cell re-mounts Google Drive (if not already mounted), creates the project folder if it doesn't exist, and then processes the `transformer_predictions` to create decision results based on a confidence threshold. Finally, it saves both the decision results and the transformer predictions to CSV files within the project folder.

In [ ]:
# # Expected utility / alert threshold example
# # Compute posterior predictive probabilities for test observations
# # (extend using pm.Data for production inference)

# from google.colab import drive
# drive.mount('/content/drive')
# import os

# project_folder = "/content/drive/MyDrive/Work Place Safety Insights"
# os.makedirs(project_folder, exist_ok=True)

# decision_results=transformer_predictions.copy()
# decision_results["alert"]=decision_results["confidence"]>0.70

# decision_results.to_csv(
#     f"{project_folder}/decision_layer_results.csv",
#     index=False)

# transformer_predictions.to_csv(
#     f"{project_folder}/bayesian_predictions.csv",
#     index=False)

# print("Outputs saved.")


In [ ]:
import numpy as np
import pandas as pd
import arviz as az
from scipy.special import expit
import os

project_folder = "/content/drive/MyDrive/Work Place Safety Insights"

# ---------------------------------------------------------
# 1. Extract posterior samples
# ---------------------------------------------------------

beta_samples = trace.posterior["beta"].values
plant_samples = trace.posterior["plant_effect"].values
cutpoint_samples = trace.posterior["cutpoints"].values

# Flatten chain and draw dimensions
beta_samples = beta_samples.reshape(-1, beta_samples.shape[-1])
plant_samples = plant_samples.reshape(-1, plant_samples.shape[-1])
cutpoint_samples = cutpoint_samples.reshape(
    -1,
    cutpoint_samples.shape[-1]
)

print("Beta samples:", beta_samples.shape)
print("Plant effect samples:", plant_samples.shape)
print("Cutpoint samples:", cutpoint_samples.shape)

# ---------------------------------------------------------
# 2. Test-set linear predictor eta
# ---------------------------------------------------------

plant_idx = test_df["plant_idx"].to_numpy().astype(int)

# beta_samples: [samples, features]
# X_test.T:       [features, observations]

eta_samples = beta_samples @ X_test.T

# Add plant random effects
plant_contribution = np.zeros_like(eta_samples)

known_plants = plant_idx >= 0

plant_contribution[:, known_plants] = (
    plant_samples[:, plant_idx[known_plants]]
)

eta_samples = eta_samples + plant_contribution

print("Eta samples:", eta_samples.shape)

# ---------------------------------------------------------
# 3. Convert Ordered Logistic posterior into probabilities
# ---------------------------------------------------------

# CDF values:
# sigmoid(cutpoint - eta)

cdf = expit(
    cutpoint_samples[:, :, None]
    - eta_samples[:, None, :]
)

prob_samples = []

# Severity class 0
prob_samples.append(cdf[:, 0, :])

# Intermediate classes
for k in range(1, n_classes - 1):
    prob_samples.append(
        cdf[:, k, :] - cdf[:, k - 1, :]
    )

# Final severity class
prob_samples.append(
    1.0 - cdf[:, -1, :]
)

# [samples, classes, observations]
prob_samples = np.stack(prob_samples, axis=1)

# Average over posterior samples
# → [observations, classes]

bayesian_probs = prob_samples.mean(axis=0).T

# Numerical safety
bayesian_probs = np.clip(bayesian_probs, 0, 1)

bayesian_probs = (
    bayesian_probs
    / bayesian_probs.sum(axis=1, keepdims=True)
)

print("Bayesian probabilities shape:", bayesian_probs.shape)

print("\nFirst Bayesian prediction:")
print(bayesian_probs[0])

print(
    "\nRow sum:",
    bayesian_probs[0].sum()
)

# ---------------------------------------------------------
# 4. SAVE REQUIRED NPY FILE
# ---------------------------------------------------------

np.save(
    f"{project_folder}/bayesian_probs_test.npy",
    bayesian_probs
)

print("\nSaved bayesian_probs_test.npy")

# ---------------------------------------------------------
# 5. Create proper Bayesian predictions CSV
# ---------------------------------------------------------

bayesian_predicted = np.argmax(
    bayesian_probs,
    axis=1
)

bayesian_confidence = np.max(
    bayesian_probs,
    axis=1
)

bayesian_predictions = test_df.copy()

bayesian_predictions["true_severity"] = (
    test_df["severity_label"].astype(int).values
)

bayesian_predictions["predicted_severity"] = (
    bayesian_predicted
)

bayesian_predictions["confidence"] = (
    bayesian_confidence
)

for i in range(n_classes):
    bayesian_predictions[f"prob_class_{i}"] = (
        bayesian_probs[:, i]
    )

bayesian_predictions.to_csv(
    f"{project_folder}/bayesian_predictions.csv",
    index=False
)

print("Saved proper bayesian_predictions.csv")

# ---------------------------------------------------------
# 6. Save Bayesian trace for future use
# ---------------------------------------------------------

az.to_netcdf(
    trace,
    f"{project_folder}/bayesian_trace.nc"
)

print("Saved bayesian_trace.nc")

# ---------------------------------------------------------
# Verification
# ---------------------------------------------------------

print("\nFiles created:")

for filename in [
    "bayesian_probs_test.npy",
    "bayesian_predictions.csv",
    "bayesian_trace.nc"
]:
    path = f"{project_folder}/{filename}"
    print(
        filename,
        "✓" if os.path.exists(path) else "✗"
    )

Beta samples: (2000, 20)
Plant effect samples: (2000, 12)
Cutpoint samples: (2000, 4)
Eta samples: (2000, 320)
Bayesian probabilities shape: (320, 5)

First Bayesian prediction:
[0.02663394 0.08684906 0.29378982 0.53984029 0.0528869 ]

Row sum: 0.9999999999999999

Saved bayesian_probs_test.npy
Saved proper bayesian_predictions.csv
Saved bayesian_trace.nc

Files created:
bayesian_probs_test.npy ✓
bayesian_predictions.csv ✓
bayesian_trace.nc ✓


In [ ]:
import joblib
import json

joblib.dump(
    pca,
    f"{project_folder}/bayesian_pca.joblib"
)

with open(
    f"{project_folder}/bayesian_plant_names.json",
    "w"
) as f:
    json.dump(
        [str(x) for x in plant_names],
        f,
        indent=2
    )

print("✓ Bayesian PCA saved")
print("✓ Bayesian plant mapping saved")

✓ Bayesian PCA saved
✓ Bayesian plant mapping saved


In [ ]:
import json
import joblib
import arviz as az

project_folder = "/content/drive/MyDrive/Work Place Safety Insights"

encoder.save(
    f"{project_folder}/bayesian_encoder"
)

joblib.dump(
    pca,
    f"{project_folder}/bayesian_pca.joblib"
)

az.to_netcdf(
    trace,
    f"{project_folder}/bayesian_trace.nc"
)

with open(
    f"{project_folder}/bayesian_plant_names.json",
    "w"
) as f:
    json.dump(
        [str(x) for x in plant_names],
        f,
        indent=2
    )

print("Bayesian deployment files saved.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Bayesian deployment files saved.
